# UC2 PhoBERT Vietnamese Sentiment — Metaflow Pipeline

**Platform:** Metaflow Local Mode
**Training:** Google Colab GPU T4
**Dataset:** UIT-VSFC (16,175 samples, 3 classes)
**Model:** vinai/phobert-base

---

## Cell 1: Cài thư viện + Setup Metaflow Local Mode

In [1]:
!pip install metaflow transformers==4.44.0 "datasets==2.14.0" accelerate scikit-learn torch peft==0.11.0 pandas "numpy<2.0" pyarrow -q

import os
os.environ["METAFLOW_DEFAULT_DATASTORE"] = "local"
os.environ["METAFLOW_DEFAULT_METADATA"] = "local"
os.environ["USERNAME"] = "DoanThao013"

import metaflow, torch, numpy as np
print(f"Metaflow: {metaflow.__version__}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.2/492.2 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.2/251.2 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

## Cell 2: Tạo file pipeline UC2
⚠️ File gộp load_data + train vào 1 step vì PyTorch/HuggingFace objects không pickle được giữa steps

In [2]:
%%writefile train_uc2_metaflow.py
import os
os.environ["METAFLOW_DEFAULT_DATASTORE"] = "local"
os.environ["METAFLOW_DEFAULT_METADATA"] = "local"
os.environ["USERNAME"] = "DoanThao013"

from metaflow import FlowSpec, step, Parameter
import time

class PhoBERTSentimentFlow(FlowSpec):
    lr = Parameter('lr', default=2e-5, type=float)
    batch_size = Parameter('batch_size', default=16, type=int)
    epochs = Parameter('epochs', default=3, type=int)
    max_length = Parameter('max_length', default=256, type=int)

    @step
    def start(self):
        self.start_time = time.time()
        print(f"=== UC2 PhoBERT — Metaflow (Colab GPU) ===")
        print(f"  Config: lr={self.lr}, batch={self.batch_size}, epochs={self.epochs}")
        self.next(self.load_and_train)

    @step
    def load_and_train(self):
        import numpy as np
        import torch
        import pandas as pd
        from datasets import Dataset, DatasetDict
        from transformers import (
            AutoTokenizer, AutoModelForSequenceClassification,
            TrainingArguments, Trainer,
        )
        from sklearn.metrics import accuracy_score, f1_score

        # Load dataset
        print("  Loading UIT-VSFC...")
        urls = {
            "train": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet",
            "validation": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet",
            "test": "https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback/resolve/refs%2Fconvert%2Fparquet/default/test/0000.parquet",
        }
        train_df = pd.read_parquet(urls["train"])
        val_df = pd.read_parquet(urls["validation"])
        test_df = pd.read_parquet(urls["test"])
        dataset = DatasetDict({
            "train": Dataset.from_pandas(train_df),
            "validation": Dataset.from_pandas(val_df),
            "test": Dataset.from_pandas(test_df),
        })
        print(f"  Data: {len(dataset['train'])} train, {len(dataset['test'])} test")

        # Tokenize
        tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")
        def tokenize_fn(examples):
            return tokenizer(examples["sentence"], padding="max_length",
                           truncation=True, max_length=self.max_length)
        tokenized = dataset.map(tokenize_fn, batched=True)
        tokenized = tokenized.rename_column("sentiment", "labels")

        # Remove extra columns before set_format to avoid numpy copy error
        cols_to_keep = ["input_ids", "attention_mask", "labels"]
        cols_to_remove = [c for c in tokenized["train"].column_names if c not in cols_to_keep]
        tokenized = tokenized.remove_columns(cols_to_remove)
        tokenized.set_format("torch")

        # Train
        def compute_metrics(eval_pred):
            logits, labels = eval_pred
            preds = np.argmax(logits, axis=-1)
            return {"accuracy": accuracy_score(labels, preds),
                    "f1": f1_score(labels, preds, average="weighted")}

        model = AutoModelForSequenceClassification.from_pretrained(
            "vinai/phobert-base", num_labels=3)
        training_args = TrainingArguments(
            output_dir="./results_uc2_meta",
            num_train_epochs=self.epochs,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            learning_rate=self.lr,
            eval_strategy="epoch", save_strategy="epoch",
            load_best_model_at_end=True, metric_for_best_model="accuracy",
            logging_steps=50, fp16=True, report_to="none", save_total_limit=1,
        )
        trainer = Trainer(model=model, args=training_args,
                         train_dataset=tokenized["train"],
                         eval_dataset=tokenized["validation"],
                         compute_metrics=compute_metrics)

        print("  Training PhoBERT...")
        trainer.train()

        # Evaluate
        test_results = trainer.evaluate(tokenized["test"])
        self.accuracy = test_results["eval_accuracy"]
        self.f1 = test_results["eval_f1"]
        self.pipeline_time = time.time() - self.start_time
        print(f"  Accuracy: {self.accuracy:.4f} | F1: {self.f1:.4f} | TC7: {self.pipeline_time:.1f}s")
        self.next(self.end)

    @step
    def end(self):
        print(f"{'='*50}")
        print(f"  RESULT: acc={self.accuracy:.4f}, f1={self.f1:.4f}, TC7={self.pipeline_time:.1f}s")
        print(f"  Config: lr={self.lr}, batch={self.batch_size}")
        print(f"{'='*50}")

if __name__ == '__main__':
    PhoBERTSentimentFlow()

Writing train_uc2_metaflow.py


## Cell 3: Chạy Repeat 3 lần (đo TC7)
⏱️ ~45-60 phút (3 × 15-20 phút/run)

📸 MINH CHỨNG: Chụp screenshot output 3 runs

In [3]:
!python train_uc2_metaflow.py run --lr 2e-5 --batch_size 16
!python train_uc2_metaflow.py run --lr 2e-5 --batch_size 16
!python train_uc2_metaflow.py run --lr 2e-5 --batch_size 16

Metaflow 2.19.28 executing PhoBERTSentimentFlow for user:DoanThao013
Creating local datastore in current directory (/content/.metaflow)
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
2026-05-11 18:27:44.719 Workflow starting (run-id 1778524064718183):
2026-05-11 18:27:44.742 [1778524064718183/start/1 (pid 9536)] Task is starting.
2026-05-11 18:27:45.306 [1778524064718183/start/1 (pid 9536)] === UC2 PhoBERT — Metaflow (Colab GPU) ===
2026-05-11 18:27:45.307 [1778524064718183/start/1 (pid 9536)] Config: lr=2e-05, batch=16, epochs=3
2026-05-11 18:27:45.374 [1778524064718183/start/1 (pid 9536)] Task finished successfully.
2026-05-11 18:27:45.378 [1778524064718183/load_and_train/2 (pid 9553)] Task is starting.
2026-05-11 18:27:54.259 [1778524064718183/load_and_train/2 (pid 9553)] 2026-05-11 18:27:54.259559: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

## Cell 4: TC2 Config Sweep (3 configs)
⏱️ ~45-60 phút

📸 MINH CHỨNG: Chụp screenshot TC2 results

In [4]:
!python train_uc2_metaflow.py run --lr 5e-6 --batch_size 16
!python train_uc2_metaflow.py run --lr 2e-5 --batch_size 16
!python train_uc2_metaflow.py run --lr 1e-4 --batch_size 32

Metaflow 2.19.28 executing PhoBERTSentimentFlow for user:DoanThao013
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
2026-05-11 19:03:36.540 Workflow starting (run-id 1778526216538683):
2026-05-11 19:03:36.557 [1778526216538683/start/1 (pid 18589)] Task is starting.
2026-05-11 19:03:37.098 [1778526216538683/start/1 (pid 18589)] === UC2 PhoBERT — Metaflow (Colab GPU) ===
2026-05-11 19:03:37.099 [1778526216538683/start/1 (pid 18589)] Config: lr=5e-06, batch=16, epochs=3
2026-05-11 19:03:37.179 [1778526216538683/start/1 (pid 18589)] Task finished successfully.
2026-05-11 19:03:37.188 [1778526216538683/load_and_train/2 (pid 18610)] Task is starting.
2026-05-11 19:03:47.894 [1778526216538683/load_and_train/2 (pid 18610)] 2026-05-11 19:03:47.894361: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
2026-05-11 19:0

## Cell 5: Xem kết quả + Export CSV

In [7]:
import os

os.environ["METAFLOW_DEFAULT_DATASTORE"] = "local"
os.environ["METAFLOW_DEFAULT_METADATA"] = "local"

from metaflow import Flow

try:
    flow = Flow('PhoBERTSentimentFlow')
    print(f"Found {len(list(flow.runs()))} runs")
except Exception as e:
    print(f"Error: {e}")

Found 6 runs


In [10]:
import os
from metaflow import Flow

# Cấu hình môi trường Metaflow
os.environ["METAFLOW_DEFAULT_DATASTORE"] = "local"
os.environ["METAFLOW_DEFAULT_METADATA"] = "local"

try:
    flow = Flow('PhoBERTSentimentFlow')
    print("=== UC2 PhoBERT — Metaflow Results ===")

    # In tiêu đề bảng
    header = f"{'run_id':<20} {'lr':<10} {'batch':<8} {'accuracy':<10} {'f1':<10} {'TC7(s)':<10}"
    print(header)
    print("-" * len(header))

    for run in flow.runs():
        try:
            # Truy xuất và định dạng dữ liệu từ mỗi run
            print(f"{run.id:<20} "
                  f"{run.data.lr:<10} "
                  f"{run.data.batch_size:<8} "
                  f"{run.data.accuracy:<10.4f} "
                  f"{run.data.f1:<10.4f} "
                  f"{run.data.pipeline_time:<10.1f}")
        except AttributeError:
            # Bỏ qua nếu run bị lỗi hoặc thiếu dữ liệu
            pass

except Exception as e:
    print(f"Lỗi truy xuất Flow: {e}")

=== UC2 PhoBERT — Metaflow Results ===
run_id               lr         batch    accuracy   f1         TC7(s)    
-------------------------------------------------------------------------
1778527918365675     0.0001     32       0.9267     0.9225     773.2     
1778527084708857     2e-05      16       0.9327     0.9286     827.1     
1778526216538683     5e-06      16       0.9232     0.9172     861.6     
1778525511882245     2e-05      16       0.9330     0.9297     698.0     
1778524773197427     2e-05      16       0.9311     0.9270     732.4     
1778524064718183     2e-05      16       0.9315     0.9268     701.8     


## ✅ HOÀN THÀNH UC2 Metaflow!

### Minh chứng cần lưu:
- [ ] Screenshot GPU T4 active
- [ ] Screenshot kết quả TC7 (3 runs)
- [ ] Screenshot TC2 sweep results
- [ ] Screenshot Client API output (bảng kết quả)
- [ ] File CSV (metaflow_uc2_results.csv)